# ECCOFS `avg`: Ambrose to Bermuda section and surface currents

The M/V Oleander sails between the Ambrose light (New York harbour approach) and Bermuda, crossing the shelf, the Gulf Stream and the Sargasso Sea. This notebook shows the ECCOFS ocean along that route: a vertical section of temperature, a section of the current across the route (the Gulf Stream), the surface current along and across the route, and a map of surface currents with the route on it.

It opens the whole-column best-estimate virtual Icechunk store (`eccofs-avg-best-nosub`) on [Source Cooperative](https://source.coop), cuts out the region, and uses [xroms](https://xroms.readthedocs.io) for the grid, the terrain-following depths (`z_rho`), the rotation of the velocities to east/north and the interpolation along the route (`xroms.interpll`). Plots are hvPlot.

The sections need every vertical level of `temp`, `u` and `v` in the box, so this notebook uses the `-nosub` store, where all 50 levels of a tile are one chunk. In a test, loading `temp` for this box took about 5 s from it against about 10 s from the default `eccofs-avg-best` store (single-level chunks, better for maps and point time series; see the README, "Chunking"). The surface map uses fields already loaded for the sections, so it costs no extra reads.

No credentials are needed. It needs icechunk 2.x (Python 3.12+), `xroms` 0.6.2 with `xgcm` 0.8.1 (newer xgcm releases break xroms' grid parsing) and `xesmf`, which `xroms.interpll` wraps. The fields are daily averages.

In [ ]:
# %pip install -q "xroms==0.6.2" "xgcm==0.8.1" xesmf hvplot datashader geoviews cartopy pyproj icechunk zarr

In [ ]:
import cartopy.crs as ccrs
import geoviews as gv
import holoviews as hv
import hvplot.xarray  # noqa: F401
import icechunk
import numpy as np
import xarray as xr
import xroms
import zarr
from pyproj import Geod

zarr.config.set({"async.concurrency": 32})

# whole-column chunks (all 50 levels of a tile): the better layout for sections. For maps and time series use eccofs-avg-best
URL = "https://data.source.coop/ocean-icechunks/noaa-eccofs/eccofs-avg-best-nosub"
SOURCE = "s3://noaa-nos-eccofs-pds/"   # public bucket holding the NetCDF bytes

# the route (edit me)
START = (-73.80, 40.46)   # lon, lat: Ambrose light
END = (-64.70, 32.40)     # Bermuda
WHEN = "2026-11-23 12:00" # nearest time in the store
BOX = (-76, -62, 30, 42)  # lon min, lon max, lat min, lat max: covers the route and the map

In [ ]:
# the NetCDF bytes live in NOAA's bucket, outside the store: authorize that virtual chunk container (anonymous)
repo = icechunk.Repository.open(
    icechunk.http_storage(URL),
    authorize_virtual_chunk_access=icechunk.containers_credentials({SOURCE: icechunk.s3_anonymous_credentials()}),
)
ds = xr.open_zarr(repo.readonly_session("main").store, consolidated=False, chunks={})
print(f"{ds.sizes['time']} times: {ds.time.values[0]} to {ds.time.values[-1]}")

## Cut out the region and one time

The full grid is 1443 x 1667 x 50 levels, so subset first. The rho rows and columns that touch the box become `X`/`Y` slices for `xroms.subset`, which slices the staggered `u`/`v`/`psi` dimensions consistently (one fewer point than `rho` across the staggering), as xgcm requires. It only touches dimensions that have a coordinate, and the store's horizontal dimensions have none, so index coordinates are added first.

In [ ]:
lon0, lon1, lat0, lat1 = BOX
box = ((ds.lon_rho > lon0) & (ds.lon_rho < lon1) & (ds.lat_rho > lat0) & (ds.lat_rho < lat1)).load()
rows, cols = np.flatnonzero(box.any("xi_rho")), np.flatnonzero(box.any("eta_rho"))   # rho rows / columns that touch the box

# xroms.subset slices every staggered grid consistently, but only for dimensions that have a coordinate
hdims = [d for d in ds.dims if d.startswith(("xi_", "eta_"))]
region = xroms.subset(
    ds.assign_coords({d: np.arange(ds.sizes[d]) for d in hdims}),
    X=slice(cols[0], cols[-1] + 1), Y=slice(rows[0], rows[-1] + 1),
)
fields = ["temp", "u", "v", "zeta", "h", "Cs_r", "Cs_w", "pm", "pn", "angle", "f", "mask_rho", "mask_u", "mask_v", "mask_psi"]
scalars = ["Vtransform", "Vstretching", "theta_s", "theta_b", "Tcline", "hc", "rho0", "spherical"]
sub = region[fields + scalars].sel(time=WHEN, method="nearest")
print("time:", sub.time.values, "| subset:", dict(sub.sizes))

## xroms dataset, depths and east/north velocities

xroms builds `z_rho` from `Vtransform`, `Vstretching`, `theta_s`, `theta_b` and `hc`, which are stored with the data (added back to the store by `scripts/add_scalars.py`). Two things the ECCOFS store needs: the SGRID `Conventions` attribute is dropped (xroms builds its own grid from the dimension names) and the arrays are rechunked to single chunks along the horizontal dimensions, which xgcm and `interpll` require.

`u` and `v` are components along the grid's `xi` and `eta` directions, on their own staggered points. `xroms.rotate_vectors` moves both to the `rho` points and rotates them by the grid `angle` to east and north.

In [ ]:
sub.attrs["Conventions"] = "CF-1.4"
sub = sub.chunk({d: -1 for d in sub.dims})
rds, xgrid = xroms.roms_dataset(sub)
z_rho = rds.z_rho
assert float(z_rho.isel(s_rho=-1).max()) < 5 and float(z_rho.isel(s_rho=0).min()) < -3000, "unexpected depths"

mask = rds.mask_rho == 1
ue, vn = xroms.rotate_vectors(rds.u, rds.v, rds.angle, xgrid=xgrid)   # east, north (m/s) at rho points
ue, vn = ue.where(mask).load(), vn.where(mask).load()
temp = rds.temp.where(mask).load()
h = rds.h.where(mask).load()
print(f"surface speed up to {float(np.hypot(ue.isel(s_rho=-1), vn.isel(s_rho=-1)).max()):.2f} m/s")

## The route

Points are spaced about 3 km (the grid size) along the great circle between `START` and `END`, with the bearing at each point. `xroms.interpll(..., which="pairs")` regrids bilinearly (with xESMF, on the sphere) to those lon/lat pairs and also interpolates `z_rho`, so the curvilinear grid needs no projection. Land is masked first. The east and north velocities are projected on the bearing: the component along the track is positive towards Bermuda (a following current for the ship), and the component across the track is positive to the left of the route (towards the north-east).

In [ ]:
geod = Geod(ellps="WGS84")
az, _, total_m = geod.inv(*START, *END)
n = int(total_m / 3000) + 1
pts = np.array([START, *geod.npts(*START, *END, n - 2), END])
lons, lats = pts[:, 0], pts[:, 1]
dist = np.linspace(0, total_m / 1000, n)                                 # km along the route
bearing = np.deg2rad(geod.inv(lons[:-1], lats[:-1], lons[1:], lats[1:])[0])
bearing = np.append(bearing, bearing[-1])                                # compass bearing at each point
print(f"{n} points over {dist[-1]:.0f} km, initial bearing {az:.0f} degrees")

In [ ]:
%%time
sec_t = xroms.interpll(temp, lons, lats, which="pairs")      # (s_rho, locations) with z_rho along the section
sec_u = xroms.interpll(ue, lons, lats, which="pairs")
sec_v = xroms.interpll(vn, lons, lats, which="pairs")
sec_h = xroms.interpll(h, lons, lats, which="pairs")

along = sec_u * np.sin(bearing) + sec_v * np.cos(bearing)    # m/s, positive towards Bermuda
across = -sec_u * np.cos(bearing) + sec_v * np.sin(bearing)  # m/s, positive to the left of the route
# quadmesh wants x and y on the same dimensions: give the distance as a 2-D coordinate like z_rho
sec_t = sec_t.assign_coords(km=(("s_rho", "locations"), np.broadcast_to(dist, sec_t.z_rho.shape).copy()))
across = across.assign_coords(km=sec_t.km)
sec_t

## Sections

Both are drawn on the true (moving) depth of each level, `z_rho`, with the sea floor from `h`. The temperature section shows the shelf water, the thermocline across the slope, the sharp Gulf Stream front and the warm Sargasso Sea water beyond. The velocity section is the current across the route: red is flow to the left of the route (towards the north-east, the direction the Gulf Stream runs) and blue is flow to the right, so the Stream appears as a red core and its recirculation as blue. Below, the surface current along and across the route.

In [ ]:
when = np.datetime_as_string(sub.time.values, unit="h")
common = dict(
    x="km", y="z_rho", rasterize=True, colorbar=True, xlabel="distance from Ambrose (km)", ylabel="depth (m)",
    frame_width=800, frame_height=300, ylim=(-2000, 5), xlim=(0, float(dist[-1])),
)
floor = hv.Area((dist, -sec_h.values, np.full(n, -4000.0)), kdims="km", vdims=["bottom", "floor"]).opts(color="#4d4d4d", line_width=0)

temp_plot = sec_t.hvplot.quadmesh(cmap="turbo", clabel="temperature (degC)", title=f"Temperature, {when} UTC", **common) * floor
vel_plot = across.hvplot.quadmesh(cmap="RdBu_r", symmetric=True, clim=(-1.5, 1.5), clabel="velocity across the route (m/s)", title="Current across the route, positive to the left (north-east)", **common) * floor

surf = xr.Dataset({"along": along.isel(s_rho=-1, drop=True), "across": across.isel(s_rho=-1, drop=True)}).assign_coords(km=dist)
surf_plot = hv.Curve((dist, surf.along.values), "km", "along").relabel("along the route (+ towards Bermuda)") * hv.Curve((dist, surf.across.values), "km", "across").relabel("across the route (+ to the left)")
surf_plot = surf_plot.opts(frame_width=800, frame_height=150, xlabel="distance from Ambrose (km)", ylabel="surface current (m/s)", xlim=(0, float(dist[-1])), legend_position="top_right", title="Surface current along and across the route")
(temp_plot + vel_plot + surf_plot).cols(1)

## Surface currents

Speed is the colour field and the arrows show direction (every 10 grid cells, about 30 km apart), over the surface level. The route is drawn in black and white.

In [ ]:
us, vs = ue.isel(s_rho=-1), vn.isel(s_rho=-1)
inside = ((rds.lon_rho > lon0) & (rds.lon_rho < lon1) & (rds.lat_rho > lat0) & (rds.lat_rho < lat1)).load()   # the subset is a rotated rectangle: clip to the box
speed = np.hypot(us, vs).where(inside).rename("speed")
speed.attrs.update(long_name="surface speed", units="m/s")
speed_plot = speed.hvplot.quadmesh(
    x="lon_rho", y="lat_rho", rasterize=True, geo=True, tiles="OSM", cmap="viridis", clim=(0, 2), alpha=0.85,
    clabel="surface speed (m/s)", frame_width=650, title=f"ECCOFS surface currents, {when} UTC",
)

step = 10   # grid cells between arrows
sel = (slice(None, None, step), slice(None, None, step))
lon_s, lat_s = (a.values[sel].ravel() for a in (rds.lon_rho, rds.lat_rho))
u_s, v_s, m_s = (a.values[sel].ravel() for a in (us, vs, speed))
ok = np.isfinite(u_s) & (m_s > 0.05)
arrows = gv.VectorField(
    (lon_s[ok], lat_s[ok], np.arctan2(v_s[ok], u_s[ok]), m_s[ok]), vdims=["angle", "speed"], crs=ccrs.PlateCarree(),
).opts(magnitude="speed", pivot="mid", scale=0.35, color="white", line_width=1)

# geoviews path with a CRS, so the degrees are projected like the map (a plain hv.Path would be read as metres, near 0,0)
track = np.c_[lons, lats]
route = gv.Path([track], crs=ccrs.PlateCarree()).opts(color="white", line_width=4) * gv.Path([track], crs=ccrs.PlateCarree()).opts(color="black", line_width=1.5)
ends = gv.Points([(*START, "Ambrose"), (*END, "Bermuda")], vdims="name", crs=ccrs.PlateCarree()).opts(color="red", size=8, tools=["hover"])
speed_plot * arrows * route * ends

The arrows are the daily average at the surface. Change `WHEN` to look at another day; `ds.time` lists the times in the store.